### Lab 8.2 Part-of-Speech Tagging

In this lab you will experiment with creating a sequence-to-sequence model for [part-of-speech (POS)](https://en.wikipedia.org/wiki/Part-of-speech_tagging) tagging on the [Brown corpus](https://en.wikipedia.org/wiki/Brown_Corpus).

The Brown corpus consists of sentences tagged with parts of speech.  Here is an example:

*Sentence:*
The Fulton County grand jury said Friday an investigation of Atlanta's recent primary election produced ``no evidence'' that any irregularities took place .

*Tags:* DET NOUN NOUN ADJ NOUN VERB NOUN DET NOUN ADP NOUN ADJ NOUN NOUN VERB . DET NOUN . ADP DET NOUN VERB NOUN .





Download the dataset:

In [1]:
# import os
# if not os.path.exists('brown_corpus'):
#     !wget "https://www.dropbox.com/scl/fi/k5q12z1do2siqk1uri80f/brown_corpus.zip?rlkey=82z1akb1d0wacpvr7mje51khf&dl=1" -O brown_corpus.zip -q
#     !unzip brown_corpus.zip

In [2]:
import os
import urllib.request
import zipfile

url = "https://www.dropbox.com/scl/fi/k5q12z1do2siqk1uri80f/brown_corpus.zip?rlkey=82z1akb1d0wacpvr7mje51khf&dl=1"
zip_path = "brown_corpus.zip"
extract_dir = "brown_corpus"

if not os.path.exists(extract_dir):
    # download
    if not os.path.exists(zip_path):
        urllib.request.urlretrieve(url, zip_path)

    # unzip
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_dir)

Load the data from pickle files:

In [3]:
import pickle

tokenized_text = pickle.load(open('brown_corpus/brown_corpus/tokens.pkl','rb'))
labels = pickle.load(open('brown_corpus/brown_corpus/labels.pkl','rb'))

`tokenized_text` is a list of lists of integers indicating the tokens in each sentence.

In [4]:
tokenized_text[0][:10]

[19304, 19707, 9944, 15509, 46802, 14723, 14582, 40619, 988, 27781]

`labels` is a corresponding list of lists of integers indicating the POS tags.

In [5]:
labels[0][:10]

[6, 3, 3, 7, 3, 9, 3, 6, 3, 4]

Calculate vocabulary size and number of labels from the dataset.

In [6]:
import numpy as np

vocab_size = int(np.max([np.max(t) for t in tokenized_text])+1)

num_labels = int(np.max([np.max(l) for l in labels])+1)
print(f'Vocabulary size: {vocab_size}\tNumber of labels: {num_labels}')

Vocabulary size: 49815	Number of labels: 13


Make a 90/10 train/test split of the dataset.

In [7]:
from sklearn.model_selection import train_test_split
tokenized_text_train, tokenized_text_test, labels_train, labels_test = train_test_split(tokenized_text,labels,test_size=0.1,random_state=42)

Here is code for a custom `Dataset` subclass that produces subsequences of the dataset.

In [8]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

class TokenDataset(Dataset):
  def __init__(self,tokenized_text,labels,max_seq_len=None):
    self.tokenized_text = tokenized_text
    self.labels = labels
    self.max_seq_len = max_seq_len
    
  def __len__(self):
    return len(self.tokenized_text)

  def __getitem__(self,idx):
    # get requested text
    token_ids = self.tokenized_text[idx]
    label_ids = self.labels[idx]

    # crop or pad as necessary
    if self.max_seq_len is not None:
      if len(token_ids)>self.max_seq_len:
        # choose random substring
        ind = np.random.randint(len(token_ids)-self.max_seq_len)
        token_ids = token_ids[ind:ind+self.max_seq_len]
        label_ids = label_ids[ind:ind+self.max_seq_len]
      else:
        # pad to maximum sequence length
        token_ids = [0]*(self.max_seq_len-len(token_ids)) + token_ids
        label_ids = [0]*(self.max_seq_len-len(label_ids)) + label_ids
    
    # return a sequence of token IDs and a label
    return torch.tensor(token_ids), torch.tensor(label_ids).long()


Create datasets and data loaders:

In [ ]:
train_ds = TokenDataset(tokenized_text_train,labels_train,max_seq_len=10)
test_ds = TokenDataset(tokenized_text_test,labels_test)

train_dl = DataLoader(train_ds,shuffle=True,batch_size=32)
test_dl = DataLoader(test_ds,shuffle=False,batch_size=1)

### Exercises

1. Grab a batch of data from `train_dl`.  Inspect the data (the shapes and values) and explain what you see.

In [10]:
x_batch, y_batch = next(iter(train_dl))

In [11]:
print(f'x_batch has shape {x_batch.shape} and values:\n\t{x_batch}')
print(f'y_batch has shape {y_batch.shape} and values:\n\t{y_batch}')

x_batch has shape torch.Size([32, 10]) and values:
	tensor([[    0,     0,     0, 43896,  5645, 10127, 43838, 41915,  2956,  8083],
        [    0,     0,     0,     0,  1139, 30406, 45769, 37289, 26536,  8083],
        [28012, 18506,  1007, 29347, 19304, 49232,  8575, 19736, 19304, 13155],
        [    0,     0,     0,     0,     0, 25146,  3262, 44532,  2956,  8083],
        [16740, 19304,   895, 27781, 23058, 21348, 35985,     5, 10127, 19304],
        [ 8122, 15509, 40402, 27781,  7499, 27388, 13638, 32664,  3805, 47803],
        [    0,     0,     0,     0,     0,     0, 11975, 44176, 27283,  7959],
        [    0,     0,     0,     0,     0,     0,     0, 17666, 32832, 35985],
        [ 9920, 29067, 27781, 41053, 30276, 16953, 11975, 39990, 27781, 23906],
        [    0,     0,     0, 19304, 35902,   511, 40059, 45092, 26655,  8083],
        [18880, 35985,  4234,  6376, 22908, 48119, 16740, 19304, 27466, 44814],
        [ 5411, 45794,  8575, 19304, 26346, 27781, 31113, 46224, 136

From the shape and contents of x_batch, it can be seen that each batch has 32 sequences with 10 tokens each. Each token is currently represented in x_batch by its token id, not its vector embedding yet. The y_batch has the same dimensions as x_batch, as it corresponds to which part of speech that each token (word) is.

2. Train a RNN model (from lab 8.1) for the POS tagging task.  Report accuracy on the test set.

Notes:
* Before the RNN, you need to use an ``nn.Embedding`` model to map tokens to vectors.
* `torch.nn.CrossEntropyLoss` and `torchmetrics.classification.Accuracy` cannot handle sequence inputs.  You will need to combine the batch and sequence dimensions using `torch.flatten` before computing the loss or computing accuracy.

In [ ]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size,output_size):
      super().__init__()
      self.hidden_size = hidden_size
      self.U = nn.Linear(input_size,hidden_size)
      self.W = nn.Linear(hidden_size,hidden_size)
      self.act = nn.SiLU()
      self.V = nn.Linear(hidden_size,output_size)
      self.embed = nn.Embedding(num_embeddings=vocab_size, embedding_dim=16)

    def forward(self,x):
      # x is a batch of sequences of shape [B,N,C]
      # output shape should be [B,N,O]

      x = self.embed(x)
      # get size of batch and sequence
      B,N,C = x.shape

      # make initial state vector
      h = torch.zeros(B,self.hidden_size).to(x.device)

      # step through sequence
      y = []
      for i in range(N):
        # apply U to input vector
        Ux = self.U(x[:,i])

        # apply W to previous state vector
        Wh = self.W(h)

        # compute new state vector
        h = self.act(Ux + Wh)

        # compute output and store
        y.append(self.V(h))

      # stack up all output vectors
      return torch.stack(y,dim=1)    

In [ ]:
device = "cuda"
# device = "cpu"

In [14]:
embed_size = 16  # used 32
model = RNN(
    input_size=embed_size,
    hidden_size=50,  # used 32
    output_size=num_labels
)
model = model.to(device)
loss_fn = torch.nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

In [ ]:
def compute_model_acc(model: nn.Sequential | nn.Module, loader: DataLoader):
    num_correct = 0
    n = 0
    
    model.eval()
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            z_batch = model(X_batch)
            z_batch = torch.flatten(z_batch, end_dim=1)
            y_batch = torch.flatten(y_batch.long(), end_dim=1)
            y_predict = torch.argmax(z_batch, dim=1)
            num_correct += torch.sum(y_predict == y_batch)
            n += len(y_batch)
        
    return num_correct / n

In [16]:
epochs = 20
for epoch in range(epochs):
    model.train()

    running_correct = 0
    total_samples = 0

    for X_batch, y_batch in train_dl:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        opt.zero_grad() # zero out the gradients
        z_batch = model(X_batch) # compute z values
        z_batch = torch.flatten(z_batch, end_dim=1)
        y_batch = torch.flatten(y_batch.long(), end_dim=1)
        loss = loss_fn(z_batch, y_batch) # compute loss
        loss.backward() # compute gradients
        opt.step() # apply gradients

        # calculate accuracy
        y_predict = torch.argmax(z_batch, dim=1)
        running_correct += (y_predict == y_batch).sum().item()
        total_samples += y_batch.size(0)

    epoch_train_acc = running_correct / total_samples
    print(f'epoch {epoch}: loss is {loss.item():.4f} -- training accuracy is {epoch_train_acc:.4f}, test accuracy is {compute_model_acc(model, test_dl):.4f}')

epoch 0: loss is 0.4802 -- training accuracy is 0.7270, test accuracy is 0.8221
epoch 1: loss is 0.2074 -- training accuracy is 0.8969, test accuracy is 0.9207
epoch 2: loss is 0.1811 -- training accuracy is 0.9397, test accuracy is 0.9427
epoch 3: loss is 0.1034 -- training accuracy is 0.9518, test accuracy is 0.9468
epoch 4: loss is 0.0791 -- training accuracy is 0.9574, test accuracy is 0.9506
epoch 5: loss is 0.0785 -- training accuracy is 0.9597, test accuracy is 0.9523
epoch 6: loss is 0.1275 -- training accuracy is 0.9609, test accuracy is 0.9529
epoch 7: loss is 0.1320 -- training accuracy is 0.9619, test accuracy is 0.9544
epoch 8: loss is 0.1404 -- training accuracy is 0.9628, test accuracy is 0.9547
epoch 9: loss is 0.1249 -- training accuracy is 0.9633, test accuracy is 0.9547
epoch 10: loss is 0.0983 -- training accuracy is 0.9638, test accuracy is 0.9546
epoch 11: loss is 0.1511 -- training accuracy is 0.9646, test accuracy is 0.9518
epoch 12: loss is 0.1231 -- training a

In [17]:
print(f'Final test accuracy is: {compute_model_acc(model, test_dl)*100:.2f}%!')

Final test accuracy is: 95.55%!
